In [1]:
# pip install pyarrow
# pip install fastparquet

In [ ]:
import numpy as np, pandas as pd, os, random, json, pickle, torch, time, itertools
from pathlib import Path
from datetime import datetime

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.use_deterministic_algorithms(True)
torch.set_float32_matmul_precision('medium')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

EXP_NAME = "additive_tags_main_global_aug"
RUN_STAMP = datetime.now().strftime("%Y_%m_%d_%H%M%S")
BASE_DIR = Path("experiments")
RUN_PATH = BASE_DIR / EXP_NAME / RUN_STAMP
CACHE_DIR = RUN_PATH / "cache"
RESULTS_DIR = RUN_PATH / "results"
PRED_DIR = RUN_PATH / "predictions"
MODELS_DIR = RUN_PATH / "models"
FLAT_DIR = CACHE_DIR / "features_flat"
SEQ_DIR = CACHE_DIR / "features_seq"
FOLDS_DIR = CACHE_DIR / "folds"

for p in [BASE_DIR, RUN_PATH, CACHE_DIR, RESULTS_DIR, PRED_DIR, MODELS_DIR, FLAT_DIR, SEQ_DIR, FOLDS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

Path("current_run.txt").write_text(f"{EXP_NAME}/{RUN_STAMP}")

SAR_LAGS = [4, 8, 12, 20]

minimal_config = {
    "seed": SEED,
    "device": str(device),
    "exp_name": EXP_NAME,
    "run_stamp": RUN_STAMP,
    "cache_layout": "per_tag_npz_bundles",
    "modalities_order": "PCSBIG",
    "sar_lags": SAR_LAGS
}
json.dump(minimal_config, open(RUN_PATH / "config.json", "w"), indent=2)

df_w = pd.read_csv("1-transaction-data/price_series.csv", index_col="date", parse_dates=True)
counts_df = pd.read_csv("1-transaction-data/counts_series.csv", index_col="date", parse_dates=True)
sent_df = pd.read_csv("2-sentiment-data/nsi_features_smooth.csv", index_col="date", parse_dates=True)
sar_df = pd.read_csv("3-satellite-data/weekly_sar_aligned.csv", index_col="date", parse_dates=True)
ir_series = pd.read_csv("4-interest-rate-data/interest_rate_series.csv", index_col="date", parse_dates=True).squeeze("columns")

sent_df = sent_df.reindex(df_w.index).ffill()
ir_series = ir_series.reindex(df_w.index).ffill()

sar_df = sar_df.reindex(df_w.index)

#nans_sar = int(sar_df.isna().sum().sum())
#if nans_sar != 0:
#    raise ValueError(f"SAR contains NaNs after alignment: {nans_sar}")

lag26_cols = [c for c in sar_df.columns if "_lag26" in c]
#if len(lag26_cols) > 0:
#    raise ValueError(f"Unexpected lag26 columns present in SAR file: {lag26_cols[:5]}")

lag20_cols = [c for c in sar_df.columns if c.endswith("_lag20")]
#if len(lag20_cols) == 0:
#    raise ValueError("No lag20 columns found in SAR file")

selected_master_projects = [
    'Business Bay','DAMAC HILLS 2','DAMAC HILLS','TOWN SQUARE','DownTown Dubai',
    'Dubai Creek Harbour','Dubai Marina','Dubai South Residential District',
    'Dubai Sports City','Al Furjan','International City Phase 1','Jumeirah Lakes Towers',
    'Jumeirah Village Circle','Jumeirah Village Triangle','Jumeriah Beach Residence  - JBR',
    'Mudon','Palm Jumeirah','Silicon Oasis','The Greens'
]

sent_cols = [c for c in sent_df.columns if c == "nsi_value" or c.startswith("nsi_") or (c.startswith("sem_pca") and c.endswith("_smooth"))]

region_targets = selected_master_projects

all_regions_for_sat = region_targets
region_sat_cols = {r:[c for c in sar_df.columns if c.startswith(f"{r}__")] for r in all_regions_for_sat}

window = 12
horizons = [2,6,10,14,18,22,26,30,34]

modalities_order = ['P','C','S','B','I','G']
order_index = {m:i for i,m in enumerate(modalities_order)}

def canonicalize_tag(tag):
    letters = sorted(list(set(tag)), key=lambda x: order_index[x])
    return ''.join(letters)

requested_tags = ['P','PC','PCS','PCSB','PCSBI','PCSBIG']
tags = []
seen = set()
for t in requested_tags:
    ct = canonicalize_tag(t)
    if ct not in seen:
        tags.append(ct)
        seen.add(ct)

size_counts = {k: 0 for k in range(1, len(modalities_order)+1)}
for t in tags:
    size_counts[len(t)] += 1
print("Tag combination summary:")
for k in range(1, len(modalities_order)+1):
    if size_counts[k] > 0:
        print(f"{k} tag{'s' if k>1 else ''} -> {size_counts[k]}")
print(f"Total tag combinations: {len(tags)}\n")

uses_global_block = any('G' in t for t in tags)

def validate_data():
    missing_price = [r for r in region_targets if r not in df_w.columns]
    missing_count = [r for r in region_targets if r not in counts_df.columns]
    if missing_price: raise ValueError(f"Missing price columns: {missing_price}")
    if missing_count: raise ValueError(f"Missing count columns: {missing_count}")
    if ir_series.isna().all(): raise ValueError("Interest rate series empty")
    if len(sent_cols) == 0: raise ValueError("No sentiment columns found")
    if uses_global_block:
        if "Global" not in df_w.columns: raise ValueError("Global price series missing but G tag is requested")
        if "Global" not in counts_df.columns: raise ValueError("Global count series missing but G tag is requested")

validate_data()

print("\n=== Data coverage ===")
def _span(idx):
    return idx.min().strftime("%Y-%m-%d"), idx.max().strftime("%Y-%m-%d"), len(idx)
for name, df_ in [("Prices",df_w),("Counts",counts_df),("Sentiment",sent_df),("SAR",sar_df)]:
    smin,smax,n=_span(df_.index)
    print(f"{name:>10}: {smin} -> {smax}  n={n}")
print("Interest rate aligned:", not ir_series.isna().any())
print("======================\n")

def build_flat_ir(tag, h, tgt):
    X,y,d=[],[],[]
    sat_cols = region_sat_cols.get(tgt, [])
    for i in range(window, len(df_w)-h):
        d.append(df_w.index[i])
        parts=[]
        if 'P' in tag: parts.append(df_w[tgt].iloc[i-window:i].values)
        if 'C' in tag: parts.append(counts_df[tgt].iloc[i-window:i].values)
        if 'S' in tag: parts.append(sent_df.iloc[i][sent_cols].values)
        if 'B' in tag and sat_cols: parts.append(sar_df.iloc[i][sat_cols].values)
        if 'I' in tag: parts.append(ir_series.iloc[i-window:i].values)
        if 'G' in tag:
            parts.append(df_w['Global'].iloc[i-window:i].values)
            parts.append(counts_df['Global'].iloc[i-window:i].values)
        X.append(np.hstack(parts)); y.append(df_w[tgt].iloc[i+h])
    return np.vstack(X), np.array(y), pd.DatetimeIndex(d)

def build_seq_ir(tag, h, tgt):
    X,y,d=[],[],[]
    sat_cols = region_sat_cols.get(tgt, [])
    for i in range(window, len(df_w)-h):
        d.append(df_w.index[i])
        blocks=[]
        if 'P' in tag: blocks.append(df_w[tgt].iloc[i-window:i].values.reshape(-1,1))
        if 'C' in tag: blocks.append(counts_df[tgt].iloc[i-window:i].values.reshape(-1,1))
        if 'S' in tag: blocks.append(sent_df.iloc[i-window:i][sent_cols].values)
        if 'B' in tag and sat_cols: blocks.append(sar_df.iloc[i-window:i][sat_cols].values)
        if 'I' in tag: blocks.append(ir_series.iloc[i-window:i].values.reshape(-1,1))
        if 'G' in tag:
            blocks.append(df_w['Global'].iloc[i-window:i].values.reshape(-1,1))
            blocks.append(counts_df['Global'].iloc[i-window:i].values.reshape(-1,1))
        X.append(np.hstack(blocks))
        y.append(df_w[tgt].iloc[i+h])
    X = np.stack(X) if len(X) > 0 else np.zeros((0,window,0))
    return X, np.array(y), pd.DatetimeIndex(d)

def make_rolling_folds(idx, n_folds=10, train_years=5, val_months=6,
                       start=pd.Timestamp('2015-09-01'), end=pd.Timestamp('2025-10-01')):
    folds=[]
    cur=start
    for _ in range(n_folds):
        train_end=cur+pd.DateOffset(years=train_years)
        val_end=train_end+pd.DateOffset(months=val_months)
        if val_end>end: break
        tr=(idx>=cur)&(idx<train_end); va=(idx>=train_end)&(idx<val_end)
        if tr.sum()>0 and va.sum()>0: folds.append((tr,va,cur,train_end,val_end))
        cur=cur+pd.DateOffset(months=val_months)
    return folds

total_combos = len(tags) * len(region_targets) * len(horizons)
saved_combo_count = 0
fail_count = 0
tick_every = 200
t0 = time.perf_counter()

anchors_rows = []
meta_records = []

for tag in tags:
    print(f"[Start] Tag: {tag}")
    flat_bundle = {}
    seq_bundle = {}
    folds_dict = {}
    per_tag_saved = 0

    for reg in region_targets:
        for h in horizons:
            try:
                Xf,yf,idx = build_flat_ir(tag,h,reg)
                flat_bundle[f"{reg}__h{h}__X"] = Xf
                flat_bundle[f"{reg}__h{h}__y"] = yf
                flat_bundle[f"{reg}__h{h}__idx"] = idx.values.astype('datetime64[ns]')

                Xs,ys,idx_s = build_seq_ir(tag,h,reg)
                seq_bundle[f"{reg}__h{h}__X"] = Xs
                seq_bundle[f"{reg}__h{h}__y"] = ys
                seq_bundle[f"{reg}__h{h}__idx"] = idx_s.values.astype('datetime64[ns]')

                if len(idx) > 0:
                    anchors_rows.append(pd.DataFrame({"tag":[tag]*len(idx),"region":[reg]*len(idx),"h":[h]*len(idx),"date":idx}))
                else:
                    anchors_rows.append(pd.DataFrame({"tag":[],"region":[],"h":[],"date":[]}))

                flds = make_rolling_folds(idx)
                folds_dict[f"{reg}__h{h}"] = flds

                meta_records.append({
                    "tag":tag,"region":reg,"horizon":h,
                    "n_rows":len(idx),
                    "n_cols_flat":int(Xf.shape[1]) if Xf.ndim==2 else 0,
                    "n_steps":window,
                    "ch_seq":int(Xs.shape[2]) if Xs.ndim==3 else 0
                })

                saved_combo_count += 1
                per_tag_saved += 1
                if saved_combo_count % tick_every == 0 or saved_combo_count == total_combos:
                    elapsed = time.perf_counter() - t0
                    print(f"Saved {saved_combo_count}/{total_combos} combos  |  elapsed {elapsed:.1f}s")
            except Exception as e:
                fail_count += 1
                print(f"[Fail] {tag} {reg} h={h}: {e}")

    np.savez_compressed(FLAT_DIR / f"{tag}.npz", **flat_bundle)
    np.savez_compressed(SEQ_DIR / f"{tag}.npz", **seq_bundle)
    pickle.dump(folds_dict, open(FOLDS_DIR / f"{tag}.pkl","wb"))
    print(f"[Done ] Tag: {tag}  | saved {per_tag_saved} region×horizon combos into bundles")

anchors_df = pd.concat(anchors_rows, ignore_index=True) if len(anchors_rows) > 0 else pd.DataFrame(columns=["tag","region","h","date"])
anchors_df.to_parquet(CACHE_DIR / "anchors.parquet", index=False)

meta_df = pd.DataFrame(meta_records)
meta_df.to_json(CACHE_DIR / "meta.json", orient="records", indent=2)

elapsed_total = time.perf_counter() - t0
print(f"\nCache summary: saved={saved_combo_count}, failed={fail_count}, total={total_combos}, elapsed={elapsed_total:.1f}s")
print(f"Bundles written to: {CACHE_DIR}")

final_config = {
    "seed": SEED,
    "device": str(device),
    "tags": tags,
    "horizons": horizons,
    "regions": region_targets,
    "window": window,
    "sar_lags": SAR_LAGS,
    "data_paths": {
        "price_series": "1-transaction-data/price_series.csv",
        "counts_series": "1-transaction-data/counts_series.csv",
        "sentiment": "2-sentiment-data/nsi_features_smooth.csv",
        "sar": "3-satellite-data/weekly_sar_aligned.csv",
        "interest_rate": "4-interest-rate-data/interest_rate_series.csv"
    },
    "exp_name": EXP_NAME,
    "run_stamp": RUN_STAMP,
    "modalities_order": "PCSBIG",
    "cache_layout": "per_tag_npz_bundles"
}
json.dump(final_config, open(RUN_PATH / "config.json", "w"), indent=2)
print(f"Data prep complete. Run folder: {RUN_PATH}")


Running on: cuda
Tag combination summary:
1 tag -> 1
2 tags -> 1
3 tags -> 1
4 tags -> 1
5 tags -> 1
6 tags -> 1
Total tag combinations: 6


=== Data coverage ===
    Prices: 2015-09-06 -> 2025-09-28  n=526
    Counts: 2015-09-06 -> 2025-09-28  n=526
 Sentiment: 2015-09-06 -> 2025-09-28  n=526
       SAR: 2015-09-06 -> 2025-09-28  n=526
Interest rate aligned: True

[Start] Tag: P
[Done ] Tag: P  | saved 171 region×horizon combos into bundles
[Start] Tag: PC
Saved 200/1026 combos  |  elapsed 10.3s
[Done ] Tag: PC  | saved 171 region×horizon combos into bundles
[Start] Tag: PCS
Saved 400/1026 combos  |  elapsed 36.5s
[Done ] Tag: PCS  | saved 171 region×horizon combos into bundles
[Start] Tag: PCSB
Saved 600/1026 combos  |  elapsed 113.5s
[Done ] Tag: PCSB  | saved 171 region×horizon combos into bundles
[Start] Tag: PCSBI
Saved 800/1026 combos  |  elapsed 225.4s
[Done ] Tag: PCSBI  | saved 171 region×horizon combos into bundles
[Start] Tag: PCSBIG
Saved 1000/1026 combos  |  elapsed 342.9